In [1]:
import polars as pl

In [3]:
itemcf_recall_path= '/home/mingyu/Recommand-System/项目/kaggle-H&M/save/candidate/itemcf_recall.parquet'
popularity_recall_week1_path='/home/mingyu/Recommand-System/项目/kaggle-H&M/save/candidate/recall_popularity_week1.parquet'
popularity_recall_week2_path='/home/mingyu/Recommand-System/项目/kaggle-H&M/save/candidate/recall_popularity_week2.parquet'
popularity_recall_week3_path='/home/mingyu/Recommand-System/项目/kaggle-H&M/save/candidate/recall_popularity_week3.parquet'
popularity_recall_week4_path='/home/mingyu/Recommand-System/项目/kaggle-H&M/save/candidate/recall_popularity_week4.parquet'
repurchase_recall_path='/home/mingyu/Recommand-System/项目/kaggle-H&M/save/candidate/recall_repurchase.parquet'
w2vec_recall_path='/home/mingyu/Recommand-System/项目/kaggle-H&M/save/candidate/recall_w2vec.parquet'

In [4]:
# 读取召回数据
itemcf_recall = pl.read_parquet(itemcf_recall_path)

popularity_recall_week1 = pl.read_parquet(popularity_recall_week1_path)
popularity_recall_week2 = pl.read_parquet(popularity_recall_week2_path)
popularity_recall_week3 = pl.read_parquet(popularity_recall_week3_path)
popularity_recall_week4 = pl.read_parquet(popularity_recall_week4_path)

repurchase_recall = pl.read_parquet(repurchase_recall_path)

w2vec_recall = pl.read_parquet(w2vec_recall_path)

In [5]:
itemcf_recall.columns,w2vec_recall.columns

(['customer_id', 'article_id', 'score'],
 ['customer_id', 'article_id', 'score'])

In [6]:
itemcf_recall = itemcf_recall.rename({"score": "itemcf_score"})
w2vec_recall = w2vec_recall.rename({"score": "w2v_score"})

In [7]:
itemcf_recall.schema

Schema([('customer_id', String),
        ('article_id', Float64),
        ('itemcf_score', Float64)])

In [8]:
w2vec_recall.schema

Schema([('customer_id', String),
        ('article_id', Int64),
        ('w2v_score', Float64)])

In [9]:
repurchase_recall.schema

Schema([('customer_id', String), ('article_id', Int64), ('rank', UInt8)])

In [10]:
repurchase_recall.schema

Schema([('customer_id', String), ('article_id', Int64), ('rank', UInt8)])

In [11]:
# itemcf召回的结果有误
itemcf_recall = itemcf_recall.with_columns(
    pl.col("article_id").cast(pl.Int64)
)

In [12]:
# 增加用于表示来源的属性
itemcf_recall = itemcf_recall.with_columns([
    pl.lit(1).alias("from_itemcf"),
])

w2vec_recall = w2vec_recall.with_columns([
    pl.lit(1).alias("from_w2vec"),
])

popularity_recall_week1 = popularity_recall_week1.with_columns([
    pl.lit(1).alias("from_popularity"),
])

popularity_recall_week2 = popularity_recall_week2.with_columns([
    pl.lit(1).alias("from_popularity"),
])

popularity_recall_week3 = popularity_recall_week3.with_columns([
    pl.lit(1).alias("from_popularity"),
])

popularity_recall_week4 = popularity_recall_week4.with_columns([
    pl.lit(1).alias("from_popularity"),
])

repurchase_recall = repurchase_recall.with_columns([
    pl.lit(1).alias("from_repurchase"),
])


In [13]:
# 丢去无用的列
popularity_recall_week1=popularity_recall_week1.drop(['rank'])
popularity_recall_week2=popularity_recall_week2.drop(['rank'])
popularity_recall_week3=popularity_recall_week3.drop(['rank'])
popularity_recall_week4=popularity_recall_week4.drop(['rank'])
repurchase_recall=repurchase_recall.drop(['rank'])

In [14]:
# 1. 把所有 recall 放到一个 list 里
dfs = [
    itemcf_recall,
    w2vec_recall,
    popularity_recall_week1,
    popularity_recall_week2,
    popularity_recall_week3,
    popularity_recall_week4,
    repurchase_recall,
]

In [15]:
fill_zero_cols = [
    "itemcf_score",
    "w2v_score",
    "from_itemcf",
    "from_w2vec",
    "from_popularity",
    "from_repurchase",
]
# 合并
data =pl.concat(dfs, how="diagonal") # 默认会将缺失的部分补成null
del dfs

In [ ]:
# 填充缺失值
ata=data.with_columns([
        pl.col(c).fill_null(0)
        for c in fill_zero_cols
    ])

In [2]:
data=pl.read_parquet('/home/mingyu/Recommand-System/项目/kaggle-H&M/save/data.parquet')

In [ ]:
# 在DuckDB（一种类似SQLite的嵌入式数据库）中对一个很大的parquet找回文件做外部去重聚合，避免内存爆炸
import duckdb

con = duckdb.connect()
con.execute("PRAGMA memory_limit='20GB'")
con.execute("PRAGMA threads=4")
con.execute("PRAGMA temp_directory='/tmp/duckdb'")
con.execute("PRAGMA enable_object_cache=false")

con.execute("""
    COPY (
        SELECT
            customer_id,
            article_id,
            MAX(itemcf_score) AS itemcf_score,
            MAX(w2v_score) AS w2v_score,
            MAX(from_itemcf) AS from_itemcf,
            MAX(from_w2vec) AS from_w2vec,
            MAX(from_popularity) AS from_popularity,
            MAX(from_repurchase) AS from_repurchase
        FROM read_parquet('data.parquet')
        GROUP BY customer_id, article_id
    )
    TO 'data_dedup_small.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);

""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
# 合并用户和文章属性
articles_path='../data/articles.parquet'
customer_path='../data/customers.parquet'
articles=pl.read_parquet(articles_path)
customers=pl.read_parquet(customer_path)

In [ ]:
data = (
    data
    .join(articles, on="article_id", how="left")
)

In [ ]:
data.write_parquet('data_with_articles.parquet')

In [ ]:
data=pl.read_parquet('data_with_articles.parquet')

In [ ]:
data = (
    data
    .join(customers, on="customer_id", how="left")
)

In [ ]:
data.write_parquet('Valid.parquet')

['customer_id',
 'article_id',
 'itemcf_score',
 'w2v_score',
 'from_itemcf',
 'from_w2vec',
 'from_popularity',
 'from_repurchase',
 'product_code',
 'product_type_no',
 'graphical_appearance_no',
 'colour_group_code',
 'perceived_colour_value_id',
 'perceived_colour_master_id',
 'department_no',
 'index_code',
 'index_group_no',
 'section_no',
 'garment_group_no',
 'FN',
 'Active',
 'fashion_news_frequency_Monthly',
 'fashion_news_frequency_NONE',
 'fashion_news_frequency_Regularly',
 'club_member_status_ACTIVE',
 'club_member_status_LEFT CLUB',
 'club_member_status_PRE-CREATE',
 'age_0',
 'age_1',
 'age_2']
